# Phát hiện gian lận thẻ tín dụng: anomaly detection và đánh giá khi cực kỳ mất cân bằng

Bộ `creditcard-fraud` (ULB): 284 807 giao dịch thẻ của khách châu Âu trong hai ngày 9/2013, chỉ
492 giao dịch gian lận (0.17%). `V1`…`V28` là thành phần PCA của feature gốc (đã ẩn danh), `Time`
là số giây từ giao dịch đầu tiên, `Amount` là số tiền, `Class` = 1 là gian lận.

Câu hỏi của bài: **không dùng nhãn**, các phương pháp anomaly detection bắt được bao nhiêu gian
lận? Và đo thế nào cho đúng khi 99.85% dữ liệu là "bình thường"?

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm. Điền vào chỗ `# TODO`.
- Cell gắn tag `explore` (dòng đầu `# explore`) đọc dữ liệu thật; test bỏ qua.
- Chấm: `uv run pytest tracks/data-science/modules/12-advanced-ml/exercises/intermediate/03-fraud-anomaly -q`

In [ ]:
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

FEATURES = [f"V{i}" for i in range(1, 29)] + ["Amount"]

In [ ]:
# explore: đọc dữ liệu thật (tải bằng: uv run python data/download_dataset.py creditcard-fraud)
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
df = pd.read_csv(ROOT / "data/raw/creditcard-fraud/creditcard.csv")
print(df.shape, "gian lận:", df.Class.sum(), f"({df.Class.mean():.3%})")
df.groupby("Class").Amount.describe()

## 1. Chia theo thời gian

`day_split(df)`: train là ngày thứ nhất (`Time < 86 400` giây), test là ngày thứ hai. Không chia
ngẫu nhiên: trong vận hành thật, model học trên quá khứ và chấm giao dịch tương lai, và kẻ gian
thay đổi hành vi theo thời gian.

In [ ]:
def day_split(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """(train, test): Time < 86 400 là train, còn lại là test."""
    # TODO
    raise NotImplementedError

## 2. Metric cho dữ liệu cực kỳ mất cân bằng

**Average precision** (AP, cách tính PR-AUC chuẩn của sklearn): sắp giao dịch theo điểm giảm dần,
mỗi ngưỡng $$n$$ (sau mỗi nhóm điểm bằng nhau) có precision $$P_n$$ và recall $$R_n$$:

$$
\mathrm{AP} = \sum_n (R_n - R_{n-1})\, P_n
$$

Một bộ chấm điểm ngẫu nhiên có AP xấp xỉ tỉ lệ dương tính (0.15% ở đây), không phải 0.5 như
ROC-AUC. Tự viết, không dùng `sklearn.metrics`. Các điểm bằng nhau phải được xét **cùng lúc** (một
ngưỡng), như sklearn.

`alert_metrics(y, score, n_alerts)`: đội điều tra mỗi ngày chỉ xem được `n_alerts` giao dịch có
điểm cao nhất. Trả `{"precision", "recall"}` của danh sách đó (bằng điểm thì giữ thứ tự xuất hiện:
`np.argsort(-score, kind="stable")`).

In [ ]:
def average_precision(y, score) -> float:
    """AP = Σ (R_n − R_{n−1}) · P_n, ngưỡng đặt sau mỗi nhóm điểm bằng nhau."""
    # TODO: sort điểm giảm dần; cộng dồn số dương tính (tp) và số phần tử
    # TODO: chỉ giữ vị trí cuối của mỗi nhóm điểm bằng nhau
    raise NotImplementedError


def alert_metrics(y, score, n_alerts: int) -> dict[str, float]:
    """precision và recall của n_alerts giao dịch điểm cao nhất."""
    # TODO
    raise NotImplementedError

## 3. Hai bộ phát hiện bất thường không dùng nhãn

- `isolation_forest_scores(X_train, X_test, seed=0, n_estimators=200)`: fit `IsolationForest` trên
  train, trả điểm trên test, **càng cao càng bất thường** (lưu ý dấu của `score_samples`).
- `pca_reconstruction_scores(X_train, X_test, n_components=15)`: một autoencoder **tuyến tính**.
  Chuẩn hoá bằng trung bình/độ lệch chuẩn của **train**, fit PCA trên train, nén test xuống
  `n_components` chiều rồi dựng lại; điểm là tổng bình phương sai số dựng lại của mỗi dòng.

Cả hai không được nhìn vào test khi fit: điểm của một giao dịch không được thay đổi khi ta thêm
hoặc bớt giao dịch khác trong test.

In [ ]:
def isolation_forest_scores(X_train, X_test, seed: int = 0, n_estimators: int = 200) -> np.ndarray:
    """Điểm bất thường (cao = bất thường) của X_test, IsolationForest fit trên X_train."""
    # TODO: IsolationForest(n_estimators=..., random_state=seed); score_samples cao = bình thường
    raise NotImplementedError


def pca_reconstruction_scores(X_train, X_test, n_components: int = 15) -> np.ndarray:
    """Sai số dựng lại (tổng bình phương) của X_test sau khi chuẩn hoá và PCA học trên X_train."""
    # TODO: StandardScaler().fit(X_train); PCA(n_components).fit(...); transform rồi inverse_transform
    raise NotImplementedError

In [ ]:
# explore: so sánh trên ngày thứ hai
train, test = day_split(df)
X_tr = train[FEATURES].assign(Amount=np.log1p(train.Amount))
X_te = test[FEATURES].assign(Amount=np.log1p(test.Amount))
y = test.Class.to_numpy()
print(f"train {len(train)} ({train.Class.sum()} gian lận) | test {len(test)} ({y.sum()} gian lận)")

from sklearn.metrics import roc_auc_score
scores = {
    "random": np.random.default_rng(0).random(len(y)),
    "Amount (càng nhỏ càng nghi?)": -X_te.Amount.to_numpy(),
    "isolation forest": isolation_forest_scores(X_tr, X_te),
    "PCA, học trên mọi giao dịch train": pca_reconstruction_scores(X_tr, X_te),
    "PCA, học trên giao dịch sạch": pca_reconstruction_scores(X_tr[train.Class.to_numpy() == 0], X_te),
}
rows = {name: {"roc_auc": roc_auc_score(y, s), "ap": average_precision(y, s),
               **{f"{k}@{n}": v for n in (100, 500) for k, v in alert_metrics(y, s, n).items()}}
        for name, s in scores.items()}
pd.DataFrame(rows).T.round(3)

In [ ]:
# explore: ROC-AUC "đẹp" nhưng PR curve nói khác. Vẽ cả hai cho isolation forest.
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, roc_curve

s = scores["isolation forest"]
fpr, tpr, _ = roc_curve(y, s)
prec, rec, _ = precision_recall_curve(y, s)
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].plot(fpr, tpr); ax[0].set(xlabel="FPR", ylabel="TPR", title="ROC")
ax[1].plot(rec, prec); ax[1].set(xlabel="recall", ylabel="precision", title="Precision-Recall")
plt.show()

In [ ]:
# explore: isolation forest có ổn định theo seed không?
pd.Series({seed: average_precision(y, isolation_forest_scores(X_tr, X_te, seed=seed)) for seed in range(5)}).round(3)

## 4. Nhận xét (viết vào cell này)

1. ROC-AUC của isolation forest trên 0.9. Vì sao con số đó gây hiểu lầm? Metric nào bạn báo cáo cho
   trưởng nhóm điều tra gian lận, và vì sao?
2. PCA học trên mọi giao dịch và PCA học trên giao dịch sạch khác nhau thế nào? Trong thực tế bạn có
   "giao dịch sạch" để học không?
3. Nếu có nhãn (dù chậm vài tuần), bạn còn dùng anomaly detection không? Dùng vào việc gì?